# NB_07_composed_inference_analysis - Composed Inference Profiling

**Purpose.** Profile the composed-model latency and call counts: per-stage timing (benchmark_composed_stages), monkey-patch-based call counters (profile_composed_call_counts), and FPS comparison against the E2E variant.

**Inputs.** loaded composed and E2E predictors, a small set of representative frames.

**Outputs.** per-stage timing DataFrame, call-count and cache-impact tables, FPS comparison bar chart.

**How to run.** Run top-to-bottom. The benchmark helpers in the Benchmark Helpers section are reusable from other notebooks.

**Position in the workflow.** Latency complement to NB_06 (which focuses on prediction quality).


In [ ]:
import os
from collections import defaultdict
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from ultralytics import YOLO

import MIREIA.perception.feature_integration as fi_mod
from MIREIA.config import Config
from MIREIA.data_collection.dataset_utils import load_jsonl_records, resolve_image_path
from MIREIA.perception import (
    DepthAnythingV2Estimator,
    FeatureIntegrator,
    QueuedComposedBDUGRURiskInference,
    QueuedE2ERiskInference,
    create_environment_classifier_predictor,
    load_road_segmentation_model,
)
from MIREIA.perception.flow import EgoMotionEstimator, track_objects

pd.set_option("display.max_columns", 200)
np.set_printoptions(suppress=True, precision=4)

print("Imports loaded.")

In [ ]:
# --- Experiment configuration ---
source_jsonl_name = "dataset.jsonl"
dataset_index = 3
start_frame = 0
profile_num_frames = 120
stage_sample_pairs = 20

scenarios_root = Path(Config.PATH_TO_SCENARIOS)
scenario_dirs = [
    p for p in sorted(scenarios_root.iterdir())
    if p.is_dir() and p.name not in {"videos", "__pycache__"} and (p / source_jsonl_name).is_file()
]

if not scenario_dirs:
    raise RuntimeError(f"No scenarios with {source_jsonl_name} found under {scenarios_root}")

if dataset_index < 0 or dataset_index >= len(scenario_dirs):
    raise ValueError(f"dataset_index={dataset_index} out of range [0, {len(scenario_dirs) - 1}]")

selected_scenario_dir = scenario_dirs[dataset_index]
records = load_jsonl_records(str(selected_scenario_dir / source_jsonl_name))
if not records:
    raise RuntimeError(f"No records found in {selected_scenario_dir / source_jsonl_name}")

end_frame = min(len(records), start_frame + int(profile_num_frames))
selected_records = records[start_frame:end_frame]
if not selected_records:
    raise RuntimeError("Selected frame range is empty.")

frame_paths = []
frame_ids = []
real_risks = []
for rec in selected_records:
    rel_image = str(rec.get("rgb_image_path", "")).strip()
    if not rel_image:
        continue

    full_path = resolve_image_path(str(selected_scenario_dir), rel_image, normalize_paths=True)
    if not os.path.isfile(full_path):
        continue

    frame_paths.append(full_path)
    frame_ids.append(int(rec.get("frame_id", len(frame_ids))))
    real_risks.append(float(rec.get("ground_truth_risk", 0.0)))

if len(frame_paths) < 4:
    raise RuntimeError("Need at least 4 valid frames for profiling.")

profile_paths = frame_paths
print(f"Selected scenario: {selected_scenario_dir.name}")
print(f"Usable profiling frames: {len(profile_paths)}")

In [ ]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_PATHS = {
    "e2e_full": Path(Config.PATH_TO_MODELS) / "e2e_risk_checkpoint.pt",
    "bdu_stable": Path(Config.PATH_TO_MODELS) / "bdu_gru_risk_checkpoint_stable.pt",
    "yolo": Path(Config.PATH_TO_MODELS) / "yolo11s.pt",
    "depth": Path(Config.PATH_TO_MODELS) / "depth_anything_v2_vits.pth",
    "climate": Path(Config.PATH_TO_MODELS) / "environment_multitask_checkpoint.pt",
    "road_seg": Path(Config.PATH_TO_MODELS) / "road_segmentation_multitask_checkpoint.pt",
}

required = {
    "e2e_full": MODEL_PATHS["e2e_full"],
    "bdu_stable": MODEL_PATHS["bdu_stable"],
    "yolo": MODEL_PATHS["yolo"],
    "depth": MODEL_PATHS["depth"],
}

missing_required = [f"{name}: {path}" for name, path in required.items() if not path.is_file()]
if missing_required:
    raise FileNotFoundError("Missing required model file(s):\n" + "\n".join(missing_required))

yolo_model = YOLO(str(MODEL_PATHS["yolo"]))
depth_estimator = DepthAnythingV2Estimator(
    checkpoint_path=MODEL_PATHS["depth"],
    encoder="vits",
    device=device_name,
)

environment_predictor = None
if MODEL_PATHS["climate"].is_file():
    environment_predictor = create_environment_classifier_predictor(
        checkpoint_path=str(MODEL_PATHS["climate"]),
        device=device_name,
    )

road_segmentation = None
if MODEL_PATHS["road_seg"].is_file():
    road_segmentation = load_road_segmentation_model(
        checkpoint_path=str(MODEL_PATHS["road_seg"]),
        device=device_name,
    )

feature_integrator = FeatureIntegrator()

e2e_predictor = QueuedE2ERiskInference.from_checkpoint(
    checkpoint_path=str(MODEL_PATHS["e2e_full"]),
    device=device_name,
)

composed_predictor = QueuedComposedBDUGRURiskInference.from_checkpoint(
    checkpoint_path=str(MODEL_PATHS["bdu_stable"]),
    feature_integrator=feature_integrator,
    yolo_model=yolo_model,
    depth_estimator=depth_estimator,
    environment_predictor=environment_predictor,
    road_segmentation=road_segmentation,
    device=device_name,
)

composed_predictor_no_aux = QueuedComposedBDUGRURiskInference.from_checkpoint(
    checkpoint_path=str(MODEL_PATHS["bdu_stable"]),
    feature_integrator=FeatureIntegrator(),
    yolo_model=yolo_model,
    depth_estimator=depth_estimator,
    environment_predictor=None,
    road_segmentation=None,
    device=device_name,
)

print(f"Device: {device_name}")
print("Loaded core models for profiling.")
print("Environment predictor loaded:", environment_predictor is not None)
print("Road segmentation loaded:", road_segmentation is not None)

## Quick Implementation Audit

Based on the current implementation:

1. The composed queue path computes a pair feature for each step using the previous and current frame.
2. Inside each pair extraction, it runs YOLO tracking twice and depth estimation twice (one per frame).
3. For a sliding window stream, this reprocesses the previous frame repeatedly.
4. Optional environment and road heads also run every step on frame2.
5. A pair-feature cache exists and helps when replaying the same frame sequence, but does not avoid recomputation on first pass for overlapping pairs.

The next cells quantify this with timings and call counts.

## Benchmark Helpers

One function per cell below.

### `_run_fps` — End-to-End FPS Timer

In [ ]:
def _run_fps(predictor, frame_paths_local, clear_cache=False):
    predictor.reset_queue()
    if clear_cache and hasattr(predictor, "clear_preprocess_cache"):
        predictor.clear_preprocess_cache()

    per_frame_ms = []
    t0 = perf_counter()
    for path in frame_paths_local:
        t1 = perf_counter()
        predictor.add_image_path(path)
        per_frame_ms.append((perf_counter() - t1) * 1000.0)

    elapsed = perf_counter() - t0
    fps = len(frame_paths_local) / max(elapsed, 1e-9)
    return {
        "elapsed_s": elapsed,
        "fps": fps,
        "mean_frame_ms": float(np.mean(per_frame_ms)),
        "p95_frame_ms": float(np.percentile(per_frame_ms, 95)),
    }


def _load_rgb(path: str) -> np.ndarray:
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.uint8)

### `benchmark_composed_stages` — Per-Stage Timing

In [ ]:
def benchmark_composed_stages(
    frame_paths_local,
    sample_pairs,
    feature_integrator_local,
    yolo_model_local,
    depth_estimator_local,
    environment_predictor_local,
    road_segmentation_local,
):
    pair_count = min(sample_pairs, len(frame_paths_local) - 1)
    if pair_count <= 0:
        raise RuntimeError("Need at least 2 frames.")

    frames = [_load_rgb(path) for path in frame_paths_local[: pair_count + 1]]
    ego_estimator = EgoMotionEstimator(crop_ratio=0.9)
    timings = defaultdict(list)

    for i in range(pair_count):
        frame1, frame2 = frames[i], frames[i + 1]

        for key, fn in [
            ("yolo_track_f1", lambda: track_objects(yolo_model_local, frame1)),
            ("yolo_track_f2", lambda: track_objects(yolo_model_local, frame2)),
            ("depth_f1", lambda: depth_estimator_local.predict(frame1)),
            ("depth_f2", lambda: depth_estimator_local.predict(frame2)),
            ("ego_motion", lambda: ego_estimator.estimate_motion(frame1, frame2)),
            ("road_seg", lambda: feature_integrator_local._infer_road_relative_size(frame_rgb=frame2, road_segmentation=road_segmentation_local)),
            ("environment", lambda: feature_integrator_local._infer_environment_probs(frame_rgb=frame2, environment_predictor=environment_predictor_local)),
        ]:
            t = perf_counter()
            result = fn()
            timings[key].append((perf_counter() - t) * 1000.0)

        if i == 0:
            _tracks1 = track_objects(yolo_model_local, frame1)
            _tracks2 = track_objects(yolo_model_local, frame2)
            _d1 = np.asarray(depth_estimator_local.predict(frame1).depth_map, dtype=np.float32)
            _d2 = np.asarray(depth_estimator_local.predict(frame2).depth_map, dtype=np.float32)
            _bgx, _bgy = ego_estimator.estimate_motion(frame1, frame2)
            _road = feature_integrator_local._infer_road_relative_size(frame_rgb=frame2, road_segmentation=road_segmentation_local)
            _day, _climate = feature_integrator_local._infer_environment_probs(frame_rgb=frame2, environment_predictor=environment_predictor_local)

            t = perf_counter()
            feature_integrator_local.extract_state_vector(
                yolo_tracks1=_tracks1, yolo_tracks2=_tracks2,
                depth_map1=_d1, depth_map2=_d2,
                bg_flow_x=_bgx, bg_flow_y=_bgy,
                road_relative_size=_road, day_prob=_day, climate_probs=_climate,
            )
            timings["feature_assembly"].append((perf_counter() - t) * 1000.0)

    rows = []
    for stage, values in timings.items():
        arr = np.asarray(values, dtype=np.float64)
        rows.append({"stage": stage, "mean_ms": arr.mean(), "median_ms": float(np.median(arr)), "p95_ms": float(np.percentile(arr, 95))})

    return pd.DataFrame(rows).sort_values("mean_ms", ascending=False)

### `profile_composed_call_counts` — Call Counter via Monkey-Patching

In [ ]:
def profile_composed_call_counts(predictor, frame_paths_local, clear_cache):
    counters = defaultdict(int)
    timers = defaultdict(float)

    orig_track_objects = fi_mod.track_objects
    orig_depth_predict = predictor.depth_estimator.predict
    orig_extract = predictor.feature_integrator.extract_state_vector_from_sources

    def wrapped_track_objects(*args, **kwargs):
        t = perf_counter()
        out = orig_track_objects(*args, **kwargs)
        timers["track_objects_s"] += perf_counter() - t
        counters["track_objects_calls"] += 1
        return out

    def wrapped_depth_predict(source):
        t = perf_counter()
        out = orig_depth_predict(source)
        timers["depth_predict_s"] += perf_counter() - t
        counters["depth_predict_calls"] += 1
        return out

    def wrapped_extract(*args, **kwargs):
        t = perf_counter()
        out = orig_extract(*args, **kwargs)
        timers["feature_extract_total_s"] += perf_counter() - t
        counters["feature_extract_calls"] += 1
        return out

    fi_mod.track_objects = wrapped_track_objects
    predictor.depth_estimator.predict = wrapped_depth_predict
    predictor.feature_integrator.extract_state_vector_from_sources = wrapped_extract

    try:
        predictor.reset_queue()
        if clear_cache:
            predictor.clear_preprocess_cache()

        per_frame_ms = []
        t0 = perf_counter()
        for path in frame_paths_local:
            t1 = perf_counter()
            _ = predictor.add_image_path(path)
            per_frame_ms.append((perf_counter() - t1) * 1000.0)
        elapsed = perf_counter() - t0
    finally:
        fi_mod.track_objects = orig_track_objects
        predictor.depth_estimator.predict = orig_depth_predict
        predictor.feature_integrator.extract_state_vector_from_sources = orig_extract

    return {
        "frames": len(frame_paths_local),
        "elapsed_s": float(elapsed),
        "fps": float(len(frame_paths_local) / max(elapsed, 1e-9)),
        "mean_frame_ms": float(np.mean(per_frame_ms)),
        "track_objects_calls": int(counters["track_objects_calls"]),
        "depth_predict_calls": int(counters["depth_predict_calls"]),
        "feature_extract_calls": int(counters["feature_extract_calls"]),
        "track_objects_s": float(timers["track_objects_s"]),
        "depth_predict_s": float(timers["depth_predict_s"]),
        "feature_extract_total_s": float(timers["feature_extract_total_s"]),
    }

## Stage-Level Profiling

In [ ]:
stage_df = benchmark_composed_stages(
    frame_paths_local=profile_paths,
    sample_pairs=stage_sample_pairs,
    feature_integrator_local=feature_integrator,
    yolo_model_local=yolo_model,
    depth_estimator_local=depth_estimator,
    environment_predictor_local=environment_predictor,
    road_segmentation_local=road_segmentation,
)

print(f"Pair samples profiled: {min(stage_sample_pairs, len(profile_paths) - 1)}")
display(stage_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=stage_df, x="median_ms", y="stage", color="#2a9d8f")
plt.xlabel("Median Stage Time (ms)")
plt.ylabel("Stage")
plt.title("Composed Pipeline Stage Cost (Per Pair)")
plt.tight_layout()
plt.show()

## Call-Count and Cache Impact Analysis

In [ ]:
first_pass = profile_composed_call_counts(predictor=composed_predictor, frame_paths_local=profile_paths, clear_cache=True)
second_pass = profile_composed_call_counts(predictor=composed_predictor, frame_paths_local=profile_paths, clear_cache=False)

count_df = pd.DataFrame([
    {"pass": "first_pass_cold_cache", **first_pass},
    {"pass": "second_pass_warm_cache", **second_pass},
])
count_df["track_calls_per_frame"] = count_df["track_objects_calls"] / count_df["frames"]
count_df["depth_calls_per_frame"] = count_df["depth_predict_calls"] / count_df["frames"]
count_df["feature_extract_calls_per_frame"] = count_df["feature_extract_calls"] / count_df["frames"]

display(count_df)

print("\nInterpretation hints:")
print("- Near 2.0 track/depth calls per frame → double per-step reprocessing of prev frame.")
print("- Near 0.0 calls on warm pass → pair/source cache is effective.")

## FPS Comparison — E2E vs Composed Variants

In [ ]:
bench_rows = [
    {"pipeline": "E2E", **_run_fps(e2e_predictor, profile_paths, clear_cache=True)},
    {"pipeline": "Composed (cold cache)", **_run_fps(composed_predictor, profile_paths, clear_cache=True)},
    #{"pipeline": "Composed (warm cache)", **_run_fps(composed_predictor, profile_paths, clear_cache=False)},
    {"pipeline": "Composed no env/road (cold)", **_run_fps(composed_predictor_no_aux, profile_paths, clear_cache=True)},
]

fps_df = pd.DataFrame(bench_rows).sort_values("fps", ascending=False)
display(fps_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=fps_df, x="fps", y="pipeline", color="#264653")
plt.xlabel("Frames Per Second")
plt.ylabel("Pipeline")
plt.title("Queued Inference FPS Comparison")
plt.tight_layout()
plt.show()

base_composed_fps = float(fps_df.loc[fps_df["pipeline"] == "Composed (cold cache)", "fps"].iloc[0])
base_e2e_fps = float(fps_df.loc[fps_df["pipeline"] == "E2E", "fps"].iloc[0])
print(f"E2E / Composed(cold) speed ratio: {base_e2e_fps / max(base_composed_fps, 1e-9):.2f}x")
